# 04 - Cloud Detection, Semantic Masking & GAN Guidance Parameters

This notebook bridges the gap between **Synthetic Cloud Generation** and **Downstream GAN Inpainting**:
1. Detects clouds directly from 3-band LISS-IV satellite imagery ($B_2$ Green, $B_3$ Red, $B_4$ NIR).
2. Generates continuous cloud density & opacity masks $\hat{M} \in [0, 1]$.
3. Computes quantitative conditioning parameters:
   - **Cloud Coverage Ratio (CCR)**
   - **Mean Cloud Optical Thickness / Opacity**
   - **Thick vs. Thin Cloud breakdown** (for deciding between dehazing vs generative inpainting)
4. Constructs a **4-Channel Guidance Tensor** `[B2, B3, B4, Cloud_Mask]` ready to feed directly into the downstream cGAN / Pix2Pix Generator.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.cloud_detector import SpectralCloudDetector, extract_cloud_parameters, to_fcc, preview_detection

DATA_DIR = PROJECT_ROOT / "Data"
CLEAR_DIR = DATA_DIR / "Clear_Patches"
CLOUDY_DIR = DATA_DIR / "Cloudy_Patches"

clear_files = sorted(list(CLEAR_DIR.glob("*.npy")))
cloudy_files = sorted(list(CLOUDY_DIR.glob("*.npy")))
print(f"Total patch pairs available: {len(cloudy_files)}")


### 1. Initialize Cloud Detector & Test on Random Sample


In [ ]:
detector = SpectralCloudDetector(
    brightness_threshold=0.40,
    whiteness_tolerance=0.22,
    ndvi_max_cloud=0.20
)

# Pick a random cloudy patch
sample_idx = np.random.randint(0, len(cloudy_files))
cloudy_p = np.load(cloudy_files[sample_idx])
clear_p = np.load(clear_files[sample_idx])

# Predict continuous cloud mask
predicted_mask = detector.predict_mask(cloudy_p)

# Extract conditioning parameters for GAN
params = extract_cloud_parameters(cloudy_p, predicted_mask)

print(f"--- Cloud Parameters for {cloudy_files[sample_idx].name} ---")
for k, v in params.items():
    if k not in ["binary_mask", "guidance_tensor"]:
        print(f"  {k:28s}: {v}")
print(f"  Guidance Tensor Shape       : {params['guidance_tensor'].shape} (4-channel GAN input)")


### 2. Multi-Panel Visual Inspection
Visualizing: (1) Cloudy Input, (2) Predicted Mask, (3) Mask Overlay, (4) Ground Truth Clear Target:


In [ ]:
preview_detection(
    cloudy_patch=cloudy_p,
    cloud_mask=predicted_mask,
    clear_patch=clear_p,
    show_plot=True
)


### 3. Thick vs Thin Cloud Separation for Conditional GAN
- **Thin Clouds (Haze)**: Ground texture is partially visible $\rightarrow$ GAN performs contrast enhancement / dehazing.
- **Thick Clouds (Cumulus)**: Ground texture is 100% occluded $\rightarrow$ GAN performs contextual generative inpainting.


In [ ]:
# Compare 3 random patches with varying cloud densities
fig, axes = plt.subplots(3, 3, figsize=(12, 11))

for row in range(3):
    idx = np.random.randint(0, len(cloudy_files))
    cl = np.load(cloudy_files[idx])
    mask = detector.predict_mask(cl)
    p = extract_cloud_parameters(cl, mask)
    
    # Col 0: Input FCC
    axes[row, 0].imshow(to_fcc(cl))
    axes[row, 0].set_title(f"Input FCC [{cloudy_files[idx].name}]")
    axes[row, 0].axis("off")
    
    # Col 1: Mask Heatmap
    im = axes[row, 1].imshow(mask, cmap="magma", vmin=0, vmax=1)
    axes[row, 1].set_title(f"Mask: {p['category']}\nCoverage: {p['cloud_coverage_percent']}% | Opacity: {p['mean_cloud_opacity']}")
    axes[row, 1].axis("off")
    
    # Col 2: Inpainting Guidance Overlay
    overlay = to_fcc(cl).copy()
    overlay[mask >= 0.20, 0] = np.clip(overlay[mask >= 0.20, 0] + 0.35, 0, 1)
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f"GAN Guidance Region\nThick: {p['thick_cloud_fraction']}% | Thin: {p['thin_cloud_fraction']}%")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()
